# Census MTIS 制造业合计库存销售比：三年原始来源样本

冻结完整Stage1 f696+b472父方案，0bp费用后SOXX仍是首个经济硬门。本卡只核对2011/2018/2025年7月原始月报的制造业合计库存销售比、修订口径和美国发布时钟；不读SOXX收益，不拟合或运行账户。

In [1]:
import datetime,hashlib,io,json,os,pathlib,re,shlex,subprocess,sys,tarfile,tempfile,urllib.request,urllib.parse
from bs4 import BeautifulSoup
from pypdf import PdfReader
import qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023';snap=root/'data/snapshots/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023';snap.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists();budget=plan['budget']
state={'status':'source_capturing','plan_sha256':sha(card/'plan.json'),'GETs_started':0,'model_fits':0,'native_accounts':0,'SOXX_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[]
def capture(name,url,limit):
 state['GETs_started']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');r={'name':name,'url':url,'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'status':None,'error':None,'body_bytes':0,'body_sha256':None,'local_path':None}
 try:
  req=urllib.request.Request(url,headers={'User-Agent':'soxl-qlib-source-research'})
  with urllib.request.urlopen(req,timeout=20) as response:
   r['status']=response.status;r['final_url']=response.url;r['headers']={k:v for k,v in response.headers.items() if k.lower() in ('content-type','content-length','date','last-modified','etag')};body=response.read(limit+1)
  if len(body)>limit:raise ValueError('frozen size cap exceeded')
  p=snap/(name+'.response');p.write_bytes(body);r.update(body_bytes=len(body),body_sha256=sha(p),local_path=str(p.relative_to(root)))
 except Exception as e:r['error']=type(e).__name__+': '+str(e)[:250]
 r['ended_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat();receipts.append(r);return r
index=capture('official_index',plan['sources']['official_index_url'],budget['max_index_bytes']);pdfs=[capture(c['economic_month'],c['url'],budget['max_PDF_bytes_each']) for c in plan['sources']['frozen_original_PDFs']];(snap/'response_receipts.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
index_links=set()
if index['status']==200 and index['local_path']:
 soup=BeautifulSoup((root/index['local_path']).read_bytes(),'html.parser');index_links={urllib.parse.urljoin(plan['sources']['official_index_url'],a.get('href')) for a in soup.find_all('a') if a.get('href')}
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1);calendar=[str(x)[:10] for x in D.calendar(start_time='2011-09-01',end_time='2025-10-31',freq='day')]
case_results=[];month_order=['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER']
for case,receipt in zip(plan['sources']['frozen_original_PDFs'],pdfs):
 c={'economic_month':case['economic_month'],'url':case['url'],'index_link_exact':case['url'] in index_links,'PDF_SHA256':receipt['body_sha256'],'PDF_bytes':receipt['body_bytes'],'PDF_signature_ok':False,'printed_release_date':None,'printed_release_time_ET':None,'first_Qlib_trade_strictly_after_release':None,'manufacturers_adjusted_current_ratio_preliminary':None,'manufacturers_adjusted_previous_ratio_revised':None,'status':'unknown'}
 if receipt['status']==200 and receipt['local_path'] and c['index_link_exact']:
  data=(root/receipt['local_path']).read_bytes();c['PDF_signature_ok']=data.startswith(b'%PDF')
  if c['PDF_signature_ok']:
   reader=PdfReader(io.BytesIO(data));text='\n'.join(page.extract_text() or '' for page in reader.pages);head=text[:1600].upper();m=re.search(r'(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+(\d{1,2}),\s+(20\d{2})',head);time_ok=bool(re.search(r'10:00\s*(?:A\.?M\.?)?\s*E(?:D|S)T',head))
   if m and time_ok:
    release=f'{int(m.group(3)):04d}-{month_order.index(m.group(1))+1:02d}-{int(m.group(2)):02d}';c['printed_release_date']=release;c['printed_release_time_ET']='10:00';c['first_Qlib_trade_strictly_after_release']=next((d for d in calendar if d>release),None)
   start_match=re.search(r'(?im)^\s*Adjusted\s*\d*\s*$',text);adjusted=text[start_match.end():].split('Not Adjusted',1)[0] if start_match else '';matches=list(re.finditer(r'Manufacturers(?:²|3|2)?[^\d]{0,100}?((?:\d[\d,]*\s+){6})(\d+\.\d\d)\s+(\d+\.\d\d)\s+(\d+\.\d\d)',adjusted,re.I|re.S))
   if len(matches)==1:
    mm=matches[0];c['manufacturers_adjusted_current_ratio_preliminary']=float(mm.group(2));c['manufacturers_adjusted_previous_ratio_revised']=float(mm.group(3));c['same_release_prior_year_ratio']=float(mm.group(4))
   else:c['parse_diagnostic']={'adjusted_manufacturers_matches':len(matches),'adjusted_chars':len(adjusted)}
   if c['printed_release_date'] and c['first_Qlib_trade_strictly_after_release'] and c['manufacturers_adjusted_current_ratio_preliminary'] is not None:c['status']='official_original_pilot_semantics_and_clock_qualified'
 case_results.append(c)
result={'status':'three_year_Census_MTiS_manufacturers_ratio_source_pilot_pass' if all(c['status'].endswith('qualified') for c in case_results) else 'three_year_Census_MTiS_source_pilot_incomplete','plan_sha256':sha(card/'plan.json'),'GETs_started':state['GETs_started'],'cases':case_results,'qualified_cases':sum(c['status'].endswith('qualified') for c in case_results),'full_monthly_panel_qualified':False,'semiconductor_specific':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=sorted(p for p in snap.iterdir() if p.is_file());entries={str(p.relative_to(root)):sha(p) for p in files}
with tempfile.TemporaryDirectory(prefix='mtis-source-') as tmp:
 archive=pathlib.Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='mtis-source-restore-') as back:
  fetched=pathlib.Path(back)/'source.tar.gz'
  with fetched.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True,timeout=90)
  assert sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as t:
   members={m.name:m for m in t.getmembers() if m.isfile()};assert set(members)==set(entries)
   for rel,h in entries.items():assert hashlib.sha256(t.extractfile(members[rel]).read()).hexdigest()==h
source_evidence={**receipt,'nas_path':remote,'files':entries,'cold_restore_verified':True};(card/'source_evidence.json').write_text(json.dumps(source_evidence,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_evidence=source_evidence);rec.save_objects(artifact_path='analysis',cases=case_results,result=result);rec.log_metrics(qualified_cases=result['qualified_cases'],external_GETs=state['GETs_started'],source_only=1,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_evidence':'source_evidence.json','source_NAS_sha256':digest,'source_NAS_cold_restore_verified':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'qualified_cases':result['qualified_cases'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']},'limitations':['Manufacturers is a total sector aggregate, not semiconductor or customer inventories.','Three original sample PDFs do not certify full 2011-2026 coverage, value revisions or prediction information.','No Stage1 factor, 0bp funded account or SOXX success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':result['status'],'qualified':result['qualified_cases'],'GETs':state['GETs_started'],'recorder_id':rid,'source_NAS_sha256':digest}))


2026/10/23 14:39:52 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[9784:MainThread](2026-10-23 14:40:01,494) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[9784:MainThread](2026-10-23 14:40:01,498) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[9784:MainThread](2026-10-23 14:40:01,498) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[9784:MainThread](2026-10-23 14:40:02,537) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_manufacturer_ratio_source_pilot_20261023.


[9784:MainThread](2026-10-23 14:40:02,540) INFO - qlib.workflow - [exp.py:258] - Experiment 611275637883843576 starts running ...


[9784:MainThread](2026-10-23 14:40:02,616) INFO - qlib.workflow - [recorder.py:345] - Recorder 004171211df1415aa35f6cf84e1c1c3a starts running under Experiment 611275637883843576 ...


[9784:MainThread](2026-10-23 14:40:03,410) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "three_year_Census_MTiS_source_pilot_incomplete", "qualified": 0, "GETs": 4, "recorder_id": "004171211df1415aa35f6cf84e1c1c3a", "source_NAS_sha256": "79fce1f4120782f84415ffeef3e42b59066887cd9825eca9da4bf91d9f2ae119"}


# 来源执行结论

原固定目录请求HTTP500、三个PDF成功并已NAS保存；其值因目录门失败保持unknown。单次目录传输修复及后续精确尺寸修复见各自卡，三年份原稿语义最终在 saved_index_size_repair 核验通过。

未读SOXX收益，未拟合或运行账户。
